Rizqy Jauhary Atsaany  
235150300111038  
TKOM - Embedded Artificial Intelligence - B  


## Install dependency

In [18]:
#! pip install "everywhereml>=0.2.32"

## Load Dataset

In [19]:
"""
Get X and y data from CSV
Replace with your own columns if needed
"""
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler, LabelEncoder

def load_Xy():
  """
  Load features and labels from the CSV in the Datasets folder.
  Adjust `features` and `label_col` if your CSV uses different names.
  """
  csv_path = "../Datasets/DATA TRAINING.csv"
  df = pd.read_csv(csv_path)
  features = ['RMSSD (ms)', 'SDNN (ms)', 'BPM', 'Num R-peaks']
  label_col = 'Label Stres'
  X = df[features].values
  y = df[label_col].values
  return X, y

def get_Xy():
  """
  Normalize X and one-hot encode y.
  """
  X, y = load_Xy()
  X = np.asarray(X, dtype=float)
  y = np.asarray(y)
  le = LabelEncoder()
  y_int = le.fit_transform(y)
  num_classes = len(le.classes_)
  eye = np.eye(num_classes)
  X_norm = MinMaxScaler().fit_transform(X)
  y_hot = np.asarray([eye[yi] for yi in y_int], dtype=int)

  return X_norm, y_hot

## Instantiate Neural Network

In [20]:
"""
Instantiate NN for classification
Replace with your own topology
"""
import tensorflow as tf
from tensorflow.keras import layers


def instantiate_nn_for_classification(input_shape, num_classes):
  model = tf.keras.Sequential()
  model.add(layers.Dense(32, activation='relu', input_shape=input_shape))
  model.add(layers.Dense(16, activation='relu'))
  model.add(layers.Dense(num_classes, activation='softmax'))
  model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

  return model

## Split Dataset -> Train, validation, test

In [21]:
"""
Split data between train, validation and test
"""
from sklearn.model_selection import train_test_split


X, y = get_Xy()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.3)

## Train Model

In [24]:
"""
Train model
"""
input_shape = X.shape[1:]
num_classes = y.shape[1]

model = instantiate_nn_for_classification(input_shape, num_classes)
history = model.fit(X_train, y_train, epochs=150, batch_size=16, validation_data=(X_val, y_val))

Epoch 1/150


d:\Rizqy\Kuliah\Sem 6\Embedded Artificial Intelligence\Repo\Embedded_AI\.venv\lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


15/15 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step - accuracy: 0.4557 - loss: 1.0867 - val_accuracy: 0.4314 - val_loss: 1.0746
Epoch 2/150
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.4895 - loss: 1.0659 - val_accuracy: 0.5098 - val_loss: 1.0576
Epoch 3/150
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.5105 - loss: 1.0458 - val_accuracy: 0.5098 - val_loss: 1.0394
Epoch 4/150
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.5485 - loss: 1.0270 - val_accuracy: 0.5196 - val_loss: 1.0246
Epoch 5/150
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.5654 - loss: 1.0097 - val_accuracy: 0.5294 - val_loss: 1.0064
Epoch 6/150
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.5612 - loss: 0.9913 - val_accuracy: 0.5294 - val_loss: 0.9895
Epoch 7/150
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.5696 - loss: 0.9732 - val_accuracy: 0.5490 - val_loss: 0.9735
Epoch 8/150
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.5612 - loss: 0.9547 - val_accuracy: 0.5588 - val_loss: 0

## Export Model to C++  
After this, copy-paste the generated code to a file with .h extension for the Microcontroller

In [25]:
"""
Export NN to C++
Copy-paste the generated code inside a file named model.h or irisModel.h
in your Arduino project
"""
from everywhereml.code_generators.tensorflow import convert_model


c_header = convert_model(model, X, y, model_name='irisModel')
print(c_header)

INFO:tensorflow:Assets written to: C:\Users\ASUSTU~1\AppData\Local\Temp\tmpnkb2o5a4\assets


INFO:tensorflow:Assets written to: C:\Users\ASUSTU~1\AppData\Local\Temp\tmpnkb2o5a4\assets


Saved artifact at 'C:\Users\ASUSTU~1\AppData\Local\Temp\tmpnkb2o5a4'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 4), dtype=tf.float32, name='keras_tensor_35')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  2215496528496: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2215496526032: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2215496531312: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2215495707184: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2215495706128: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2215497077632: TensorSpec(shape=(), dtype=tf.resource, name=None)
#pragma once

#ifdef __has_attribute
#define HAVE_ATTRIBUTE(x) __has_attribute(x)
#else
#define HAVE_ATTRIBUTE(x) 0
#endif
#if HAVE_ATTRIBUTE(aligned) || (defined(__GNUC__) && !defined(__clang__))
#define DATA_ALIGN_ATTRIBUTE __attribute__((aligned(4)))
#else
#define DATA_ALIGN_ATTR